# nano-dsv4.1f — prepare 8K mid-training + SFT data

This notebook covers stages **2 and 3** of the canonical pipeline:

1. pretrain — prepared separately by the 3B-token pretraining-only notebook;
2. **midtrain** — one Q-aware curated document stage plus a small reasoning/agent mixture;
3. **SFT** — assistant-only supervision over cleaned reasoning/agent traces.

It deliberately does **not** rebuild general pretraining data. Trace shards contain both causal-LM metadata and `sft_loss_mask`, so the same cleaned bytes can serve mid-training and SFT.


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys

TOKENIZER_DATASET = 'xiayicheng3gmailcom/nano-dsv41f-tokenizer-fineweb'
REPO_URL = 'https://github.com/xiayicheng3-code/nano-dsv4.1f.git'
REPO_REF = 'codex/three-stage-training'
WORK = Path('/kaggle/working')
REPO_DIR = WORK / 'nano-dsv4.1f'
MIDTRAIN_OUT = WORK / 'nano-dsv41f-midtrain-8k'
TRACE_OUT = WORK / 'nano-dsv41f-traces-8k'

BUILD_MIDTRAIN = True
BUILD_TRACES = True
SEQ_LEN = 8192
MIDTRAIN_STEPS = 10_000
QUERY_BUDGET = 128
Q_THRESHOLD = 640
Q_BANDS = '640,768,1024,1536,2048,3072,4096,6144,8192'
REASONING_TARGET_TOKENS = 4_000_000
AGENT_TARGET_TOKENS = 8_000_000
SEED = 1701

os.environ['TOKENIZERS_PARALLELISM'] = 'true'
os.environ['RAYON_NUM_THREADS'] = str(max(1, os.cpu_count() or 1))
print({'cpu_threads': os.environ['RAYON_NUM_THREADS'], 'seq_len': SEQ_LEN, 'stage': 'midtrain+sft'})


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'kagglehub'], check=True)
import kagglehub
tokenizer_root = Path(kagglehub.dataset_download(TOKENIZER_DATASET))
candidates = sorted(tokenizer_root.rglob('tokenizer.json'))
if not candidates:
    raise FileNotFoundError(f'No tokenizer.json found in {tokenizer_root}')
TOKENIZER_PATH = candidates[0]
print('tokenizer:', TOKENIZER_PATH)


In [ ]:
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(REPO_DIR)], check=True)
commit = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{REPO_DIR}[data]'], check=True)
env = os.environ.copy()
env['PYTHONPATH'] = str(REPO_DIR / 'src') + os.pathsep + env.get('PYTHONPATH', '')
print('repo commit:', commit)


## Build the single mid-training document corpus

`MIDTRAIN_STEPS` is a budget for **mid-training only**. It is not a fraction of the 3B-token pretraining run. The stage-aware wrapper produces one phase named `midtrain`, with Q-aware packing enabled for all rows.


In [ ]:
if BUILD_MIDTRAIN:
    if MIDTRAIN_OUT.exists():
        shutil.rmtree(MIDTRAIN_OUT)
    subprocess.run([
        sys.executable, str(REPO_DIR / 'scripts/prepare_midtrain_corpus.py'),
        '--tokenizer', str(TOKENIZER_PATH),
        '--output-dir', str(MIDTRAIN_OUT),
        '--total-steps', str(MIDTRAIN_STEPS),
        '--seq-len', str(SEQ_LEN),
        '--query-budget', str(QUERY_BUDGET),
        '--q-threshold', str(Q_THRESHOLD),
        '--q-band-edges', Q_BANDS,
        '--tokenize-batch-size', '256',
        '--tokenize-batch-chars', '4000000',
        '--shard-rows', '128',
        '--seed', str(SEED),
    ], check=True, cwd=REPO_DIR, env=env)


## Build reusable reasoning/agent trace shards

The wrapper delegates cleaning/rendering/packing to the maintained trace builder and then stamps the manifest with explicit `midtrain` and `sft` views. Mid-training ignores `sft_loss_mask`; SFT applies it.


In [ ]:
if BUILD_TRACES:
    if TRACE_OUT.exists():
        shutil.rmtree(TRACE_OUT)
    subprocess.run([
        sys.executable, str(REPO_DIR / 'scripts/prepare_stage_traces.py'),
        '--tokenizer', str(TOKENIZER_PATH),
        '--output-dir', str(TRACE_OUT),
        '--pool', 'all',
        '--seq-len', str(SEQ_LEN),
        '--reasoning-target-tokens', str(REASONING_TARGET_TOKENS),
        '--agent-target-tokens', str(AGENT_TARGET_TOKENS),
        '--query-budget', str(QUERY_BUDGET),
        '--q-threshold', str(Q_THRESHOLD),
        '--q-band-edges', Q_BANDS,
        '--tokenize-batch-size', '64',
        '--shard-rows', '128',
        '--seed', str(SEED),
    ], check=True, cwd=REPO_DIR, env=env)


In [ ]:
if BUILD_MIDTRAIN:
    curriculum = json.loads((MIDTRAIN_OUT / 'curriculum_manifest.json').read_text())
    print('\nMIDTRAIN DOCUMENT MANIFEST')
    print(json.dumps(curriculum, indent=2))
    m = json.loads((MIDTRAIN_OUT / 'midtrain' / 'manifest.json').read_text())
    print(' rows:', m['packed']['rows'])
    print(' utilization:', round(m['packed']['real_token_utilization'], 4))
    print(' sources:', {k: round(v, 4) for k, v in m['phase']['source_weights_actual_real_tokens'].items()})
    print(' Q budget utilization:', round(m['query_packing']['mean_budget_utilization'], 4))
    print(' Q density:', [round(x, 6) for x in m['query_packing']['expected_q_density']])

if BUILD_TRACES:
    trace_summary = json.loads((TRACE_OUT / 'trace_manifest.json').read_text())
    print('\nTRACE STAGE VIEWS')
    print(json.dumps(trace_summary, indent=2))
    for pool in ('reasoning', 'agent'):
        m = json.loads((TRACE_OUT / pool / 'manifest.json').read_text())
        print('\n', pool)
        print(' records:', m['trace_records'], 'tokens:', m['actual_trace_tokens'], 'rows:', m['packing']['rows'])
        print(' assistant SFT fraction:', round(m['sft_supervised_fraction'], 4))
        print(' Q budget utilization:', round(m['packing']['query']['mean_budget_utilization'], 4))


## Outputs

Save these directories as Kaggle Dataset outputs:

- `/kaggle/working/nano-dsv41f-midtrain-8k`
- `/kaggle/working/nano-dsv41f-traces-8k`

The trace output is shared by mid-training and SFT; do not duplicate it just to change the loss mask.
